# Weighted A* on Real City Maps

This notebook implements the **Weighted A\*** search algorithm on real city street maps.

The `comp372_lab1_starter` package provides map loading, geocoding, and visualization utilities.

**Configuration:** Change `PLACE`, `NETWORK_TYPE`, `DIST_METERS` to switch cities.

In [123]:
# Install dependencies (run once)
import sys, subprocess

pkgs = ["osmnx", "folium", "ipywidgets", "matplotlib"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

print("Installed:", ", ".join(pkgs))

Installed: osmnx, folium, ipywidgets, matplotlib


In [124]:
from __future__ import annotations
import matplotlib as plt
import random
import heapq
import numpy as np
from typing import Dict, Tuple, List, Optional, Callable, Any
from IPython.display import display
#import logging
#logger = logging.getLogger()
#logger.setLevel(logging.DEBUG)

# Import starter package utilities
from comp372_lab1_starter import (
    generate_map_with_route,
    generate_directions,
    compute_path_cost,
    run_search_with_stats,
    print_search_result,
    SearchProblem,
    LocationNotFoundError,
    goal_test,
    heuristic_sld,
    # Testing utilities
    test_node,
    test_expand,
    test_solution,
    test_insert_and_pop,
    test_best_first_search,
)

# --------- CONFIGURATION ----------
PLACE = "Memphis, Tennessee, USA"    # <-- change this to switch cities
NETWORK_TYPE = "drive"               # "walk", "drive", "bike", etc.
DIST_METERS = 30000                  # size of region around the place center (meters)
# ----------------------------------

**Note:** The next cell may take several minutes to run the first time, as it downloads and processes the street map data for the specified region.

In [125]:
# --------- START/GOAL SELECTION ----------
START_DESCRIPTION = "Rhodes College, Memphis, TN"
GOAL_DESCRIPTION = "FedExForum, Memphis, TN"
# -----------------------------------------

try:
    problem = SearchProblem(
        start_description=START_DESCRIPTION,
        goal_description=GOAL_DESCRIPTION,
        place=PLACE,
        network_type=NETWORK_TYPE,
        dist_meters=DIST_METERS,
    )
    print(f"Start: {START_DESCRIPTION}")
    print(f"Goal: {GOAL_DESCRIPTION}")
except LocationNotFoundError as e:
    print(f"Error: {e}")

Graph loaded: 45,884 nodes, 116,187 edges

Start: Rhodes College, Memphis, TN
  Geocoded location: (35.155644, -89.988935)
  Nearest graph node: (35.155499, -89.991564)
  Distance to node: 239.5 meters

Goal: FedExForum, Memphis, TN
  Geocoded location: (35.138240, -90.050695)
  Nearest graph node: (35.138981, -90.051095)
  Distance to node: 90.1 meters

Start: Rhodes College, Memphis, TN
Goal: FedExForum, Memphis, TN


## Weighted A* Implementation

Implement the functions marked with `TODO` below to complete the best-first search algorithm.

**Provided** (do not modify):
- `problem.initial` - initial state
- `problem.goal` - goal state
- `problem.get_coordinates(state)` - (latitude, longitude)
- `problem.actions(state)` - list of available actions
- `problem.result(state, action)` - successor state resulting from action  
- `problem.action_cost(s, action, s_prime)` - action cost in meters
- `goal_test(problem, state)` - True if state is the goal
- `heuristic_sld(problem, state)` - straight-line-distance heuristic function to the goal in meters

**You must implement:**
- `Node` - the class used to encapsulate state and path information during search
- `expand(problem, node)` - expands node's successor nodes based on available actions
- `solution(node)` - generates a solution from a node
- `insert(frontier, node, f, counter)` - insert a node into a priority queue
- `pop(frontier)` - remove and return the highest priority (lowest f()) node from the priority queue
- `best_first_search(problem, f)` - an implementation of the best-first search algorithm (general search)
- `weighted_a_star(problem, heuristic, w)` - an implemention of A* based on best-first-search

In [15]:
# =============================================================================
# TODO: Implement the Node class
# =============================================================================

class Node:
    """A node in the search tree.

    Attributes:
        state: The state this node represents
        parent: The parent Node (None for the initial node)
        path_cost: The cost g(n) from initial state to this node

    Example:
        >>> root = Node(state="A")
        >>> root.state
        'A'
        >>> root.parent is None
        True
        >>> root.path_cost
        0.0
        >>> child = Node(state="B", parent=root, path_cost=5.0)
        >>> child.parent.state
        'A'
    """
    def __init__(self, state: Any, parent: Optional['Node'] = None, path_cost: float = 0.0):
        # TODO: Initialize the node attributes
        self.state = state
        self.parent = parent 
        self.path_cost = path_cost

In [16]:
# Test Node implementation
#print("hello!")
for result in test_node(Node):
    print(result)

[PASS] Node.__init__ with defaults
[PASS] Node.__init__ with all params
[PASS] Node attributes accessible


In [17]:
# =============================================================================
# TODO: Implement expand
# =============================================================================

def expand(problem: SearchProblem, node: Node) -> List[Node]:

    
    """Expand a node, generating all child nodes.

    
    For each action available from node.state:
        1. Get the resulting state: problem.result(state, action)
        2. Calculate the path cost: problem.action_cost(s, action, s_prime)
        3. Create a new Node for the child

    Args:
        problem: The search problem
        node: The node to expand

    Returns:
        List of child Nodes

    Example (using a simple graph where A connects to B with cost 1 and D with cost 4):
        >>> node = Node(state="A", path_cost=0.0)
        >>> children = expand(problem, node)
        >>> len(children)
        2
        >>> children[0].state
        'B'
        >>> children[0].path_cost
        1.0
        >>> children[0].parent.state
        'A'
    """
    children = []
    for action in problem.actions(node.state):
        childstate = problem.result(node.state, action)
        accost = problem.action_cost(node.state, action, childstate)
        accost = accost + node.path_cost
        child_node = Node(state= childstate, parent = node, path_cost = accost)
        children.append(child_node)
    return children

In [18]:
# Test expand implementation
for result in test_expand(Node, expand):
    #print(Node)
    print(result)

[PASS] expand returns correct children
[PASS] expand sets parent correctly
[PASS] expand calculates path_cost correctly
[PASS] expand accumulates path_cost
[PASS] expand handles limited successors


In [19]:
# =============================================================================
# TODO: Implement solution
# =============================================================================

def solution(node: Node) -> List[Any]:
    """Reconstruct the path from initial state to this node.

    Follow parent pointers back to the root and return the sequence of states.

    Args:
        node: The goal node

    Returns:
        List of states from initial to goal

    Example:
        >>> n1 = Node(state="A")
        >>> n2 = Node(state="B", parent=n1, path_cost=1.0)
        >>> n3 = Node(state="F", parent=n2, path_cost=5.0)
        >>> solution(n3)
        ['A', 'B', 'F']
    """
    pathli = [] 
    currnode = node
    while (currnode != None):
        pathli.append(currnode.state)
        currnode = currnode.parent
        
    return  pathli[::-1] #meant to reverse list

In [20]:
# Test solution implementation
for result in test_solution(Node, solution):
    print(result)

[PASS] solution single node
[PASS] solution two nodes
[PASS] solution longer path
[PASS] solution returns list


In [32]:
# =============================================================================
# TODO: Implement insert and pop for the frontier
# =============================================================================

def insert(frontier: List, node: Node, f: Callable[[Node], float], counter: List[int]) -> None:
    """Insert a node into the priority queue frontier.
    Args:
        frontier: The priority queue
        node: The node to insert
        f: The evaluation function f(node) -> priority value
        counter: A mutable counter [int] for tiebreaking

    Example:
        >>> frontier = []
        >>> counter = [0]
        >>> node_a = Node(state="A", path_cost=10.0)
        >>> node_b = Node(state="B", path_cost=5.0)
        >>> insert(frontier, node_a, lambda n: n.path_cost, counter)
        >>> insert(frontier, node_b, lambda n: n.path_cost, counter)
        >>> len(frontier)
        2
        >>> counter[0]
        2
    """
    # Hint: See https://docs.python.org/3/library/heapq.html

    #frontier.heappush([node, f(node)])

    #case frontier is empty
    f_val = f(node)
    heapq.heappush(frontier, (f_val, counter[0], node))
    #frontier.append((f_val, node, counter[0]))
    counter[0]+= 1
    
    
    
        

    
    
    


def pop(frontier: List) -> Node:
    """Remove and return the node with lowest f-value from the frontier.

    Args:
        frontier: The priority queue

    Returns:
        The node with the lowest f-value

    Example (continuing from insert example above):
        >>> best = pop(frontier)
        >>> best.state
        'B'
        >>> best.path_cost
        5.0
        >>> len(frontier)
        1
    """
    lowpri = heapq.heappop(frontier)
    retnode = lowpri[2]

    

    return retnode  
    # Hint: See https://docs.python.org/3/library/heapq.html

In [33]:
# Test insert and pop implementation
for result in test_insert_and_pop(Node, insert, pop):
    print(result)

[PASS] insert/pop single node
[PASS] pop returns lowest f-value first
[PASS] tiebreaking with counter
[PASS] custom f function


### Implement the BEST-FIRST-SEARCH algorithm
Here is the pseudocode for BEST-FIRST-SEARCH from AI:A Modern Approach for your convenience.

```
function BEST-FIRST-SEARCH(problem, f) returns a solution node or failure
    node ← NODE(STATE = problem.INITIAL)
    frontier ← a priority queue ordered by f, with node as an element
    reached ← a lookup table, with one entry with key problem.INITIAL and value node
    while not IS-EMPTY(frontier) do
        node ← POP(frontier)
        if problem.IS-GOAL(node.STATE) then return node
        for each child in EXPAND(problem, node) do
            s ← child.STATE
            if s is not in reached or child.PATH-COST < reached[s].PATH-COST then
                reached[s] ← child
                add child to frontier
    return failure

function EXPAND(problem, node) yields nodes
    s ← node.STATE
    for each action in problem.ACTIONS(s) do
        s' ← problem.RESULT(s, action)
        cost ← node.PATH-COST + problem.ACTION-COST(s, action, s')
        yield NODE(STATE = s', PARENT = node, ACTION = action, PATH-COST = cost)
```

In [56]:
# =============================================================================
# TODO: Implement best_first_search
# =============================================================================

def best_first_search(problem: SearchProblem, f: Callable[[Node], float]) -> Optional[Node]:
    """Find a solution node from the initial state to the goal using best-first search.

    This function explores states in order of their f-value (lowest first),
    using a priority queue as the frontier. It tracks reached states to avoid
    revisiting them unless a better path is found.

    Your implementation should use the following helper functions:
        - goal_test(problem, state): Check if a state is the goal
        - expand(problem, node): Generate child nodes
        - insert(frontier, node, f, counter): Add a node to the frontier
        - pop(frontier): Remove and return the best node from the frontier

    Args:
        problem: The search problem, providing initial/goal states and transitions
        f: Evaluation function that assigns a priority to each node

    Returns:
        The goal Node if a path exists, None otherwise.
        Call solution(node) on the returned node to get the path.
"""
    frontier = []
    counter = [0]
    initstate = problem.initial
    initnode = Node(state = initstate, path_cost = 1.0) 
    insert(frontier, initnode, f, counter)
    reached = { initstate : initnode } 

    while (frontier):
        node = pop(frontier)
        if (goal_test(problem, node.state)):
            return node
        for child in expand(problem, node): 
            s = child.state
            if ((not(s in reached)) or (child.path_cost < reached[s].path_cost)):
                reached[s] = child
                insert(frontier, child, f, counter)

    return None
                
                
            
            
        
        
        
    
    

    
    # TODO: Implement best_first_search
    pass

In [57]:
# Test best_first_search implementation
for result in test_best_first_search(Node, expand, solution, insert, pop, best_first_search):
    print(result)

[PASS] best_first_search finds path
[PASS] best_first_search finds optimal path
[PASS] best_first_search with heuristic
[PASS] best_first_search returns None for no path
[PASS] best_first_search handles start=goal


In [97]:
# =============================================================================
# TODO: Implement weighted_a_star
# =============================================================================
def weighted_a_star(
    problem: SearchProblem,
    heuristic: Callable[[SearchProblem, Any], float],
    w: float = 1.0,
) -> Optional[Node]:
    """Weighted A* using best_first_search.

    Args:
        problem: The search problem
        heuristic: Heuristic function h(problem, state) -> estimated cost to goal
        w: Weight for heuristic (w=1.0 is standard A*, w>1.0 is weighted A*)

    Returns:
        The goal Node if a path exists, None otherwise.
        Call solution(node) on the returned node to get the path.

        
    """

   # heuro = heuristic(problem, w)

    optimistic_path = heuristic(problem, problem.goal) 

   # assert type(optimistic_path) is float 
    ret = best_first_search(problem, lambda  node : node.path_cost + optimistic_path )
    #assert type(ret) is Node
    if (ret == None):
        return None
    return ret
    
    # Hint: use best_first_search with the right evaluation function f(n)
    pass

## Baseline run (w = 1.0)


In [98]:
goal_node = weighted_a_star(problem, heuristic_sld, 1.0)
if goal_node:
    path = solution(goal_node)
    cost = compute_path_cost(problem, path)
    print(f"Path found! {len(path)} nodes, {cost:,.1f} meters ({cost/1000:.2f} km)")
else:
    path = None
    print("No path found.")

Path found! 59 nodes, 7,056.9 meters (7.06 km)


In [99]:
if path:
    display(generate_map_with_route(problem, path))

In [100]:
if path:
    generate_directions(problem, path)


Turn-by-Turn Directions
  1. Head W on Snowden Avenue                                (15 m)
  2. Turn left onto University Street                        (110 m)
  3. Turn right onto Tutwiler Avenue                         (1.6 km)
  4. Bear right onto Faxon Avenue                            (319 m)
  5. Turn left onto North Watkins Street                     (134 m)
  6. Turn right onto unnamed primary_link                    (291 m)
  7. Continue straight onto North Parkway                    (2.0 km)
  8. Turn left onto North Danny Thomas Boulevard             (1.3 km)
  9. Bear right onto unnamed primary_link                    (193 m)
 10. Continue straight onto Jefferson Avenue                 (70 m)
 11. Turn left onto North 4th Street                         (117 m)
 12. Turn right onto Court Avenue                            (15 m)
 13. Turn left onto North 4th Street                         (115 m)
 14. Turn left onto Madison Avenue                           (31 m)
 15. Turn 

## Comparison of Search Algorithms (UCS, A*, and Greedy Best-First Search)

Your `best_first_search` implementation is a **general** search algorithm — the behavior depends entirely on the evaluation function `f(n)`. By choosing different weights for the heuristic, you can recover three well-known algorithms:

| Algorithm | Evaluation Function | Weight (`w`) |
|---|---|---|
| **Uniform-Cost Search (UCS)** | f(n) = g(n) | `w = 0.0` |
| **A\* Search** | f(n) = g(n) + h(n) | `w = 1.0` |
| **Greedy Best-First Search** | f(n) ≈ h(n) | `w` very large (e.g. `1000.0`) |

**Task:** Run all three algorithms on the same problem and compare them in terms of:

1. **Solution Path Cost** — How optimal is the path each algorithm finds?
2. **Nodes Expanded** — How much work does each algorithm do?
3. **Execution Time** — How fast is each algorithm?

Present your comparison using graphs, charts, or other data visualizations. Include a brief written analysis explaining the trade-offs you observe between the three algorithms. For example, which algorithm finds the best path? Which explores the fewest nodes? How do these trade-offs relate to the role of the heuristic weight?

**Hint:** You can use the provided `run_search_with_stats` function to run a search and collect statistics. It returns a `SearchResult` object with the following attributes:

- `result.found` — `True` if a path was found
- `result.path_cost` — total path cost in meters
- `result.nodes_expanded` — number of nodes expanded during search
- `result.execution_time` — time in seconds

**Example usage:**

```python
result = run_search_with_stats(
    problem, weighted_a_star, heuristic_sld, w=1.0,
    Node=Node, expand_fn=expand, solution_fn=solution,
    insert_fn=insert, pop_fn=pop, goal_test_fn=goal_test
)
print(f"Path cost: {result.path_cost:,.1f} m")
print(f"Nodes expanded: {result.nodes_expanded}")
print(f"Execution time: {result.execution_time:.4f} s")
```

In [156]:
results = []
for i in range (100): 
    sele = random.randint(1, 3) 
   
    if (sele == 1):  
            r = random.random()
            results.append((run_search_with_stats(
    problem, weighted_a_star, heuristic_sld, w= r,
    Node=Node, expand_fn=expand, solution_fn=solution,
    insert_fn=insert, pop_fn=pop, goal_test_fn=goal_test), "Uniform Cost", r))
           # print("uni")
    elif (sele == 2):
            r = random.randrange(1, 4)
            results.append((run_search_with_stats(
    problem, weighted_a_star, heuristic_sld, w= r,
    Node=Node, expand_fn=expand, solution_fn=solution,
    insert_fn=insert, pop_fn=pop, goal_test_fn=goal_test), "A* cost", r))
           # print("astar")
    else:
            r =  random.randrange(4, 10000)
            results.append((run_search_with_stats(
    problem, weighted_a_star, heuristic_sld, w= r,
    Node=Node, expand_fn=expand, solution_fn=solution,
    insert_fn=insert, pop_fn=pop, goal_test_fn=goal_test), "Greedy", r))
            #print("greed")
for entry in results: 
        print("results category:", entry[1])
        print(f"Nodes expanded: {entry[0].nodes_expanded}")
        print(f"Execution time: {entry[0].execution_time:.4f} s")
        print(f"Path cost: {entry[0].path_cost:,.1f} m")
        print(f"Weight : {entry[2]}")
        print(f"Found way to goal : {entry[0].found}")
        print()
#"""
#plt.style.use("_mpl_gallery")

#nodesreg = [[], [], []]
#timesreg = [[], [], []]
#for i in range(3): 
 #   for c in range(len(results[i])):
  #      nodesreg[i][c] = results[i][c].nodes_expanded
   #     timesreg[i][c] = results[i][c].execution_time
   #"""     

        
#fig, ax = plt.subplots()
#3ax.hist2d(nodesreg, timesreg, bins=(np.arange(-3, 3, 0.1), np.arange(-3, 3, 0.1)))
#.set(xlim=(-2, 2), ylim=(-3, 3))

#plt.show()


results category: Uniform Cost
Nodes expanded: 1554
Execution time: 0.0153 s
Path cost: 7,056.9 m
Weight : 0.76804100872233
Found way to goal : True

results category: Uniform Cost
Nodes expanded: 3071
Execution time: 0.0344 s
Path cost: 7,056.9 m
Weight : 0.37399502183724787
Found way to goal : True

results category: Greedy
Nodes expanded: 51
Execution time: 0.0008 s
Path cost: 7,914.8 m
Weight : 5208
Found way to goal : True

results category: A* cost
Nodes expanded: 60
Execution time: 0.0008 s
Path cost: 7,663.6 m
Weight : 3
Found way to goal : True

results category: Uniform Cost
Nodes expanded: 2825
Execution time: 0.0269 s
Path cost: 7,056.9 m
Weight : 0.44219762246299377
Found way to goal : True

results category: A* cost
Nodes expanded: 60
Execution time: 0.0008 s
Path cost: 7,663.6 m
Weight : 3
Found way to goal : True

results category: Greedy
Nodes expanded: 51
Execution time: 0.0006 s
Path cost: 7,914.8 m
Weight : 5475
Found way to goal : True

results category: Greedy
Nod